In [1]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
from datetime import date

import matplotlib.pyplot as plt 
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter

import eurostat #python wrapper for taking data. 
import time

In [2]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Code\LNG terminals'

In [3]:
import os
os.chdir(Share_point)

In [4]:
os.getcwd()

'C:\\Users\\giovanni.sgaravatti\\Bruegel\\Research - 2021-11 European natural gas imports\\Code\\LNG terminals'

In [5]:
Points_APIs = pd.read_excel(r'Dataproviders 09.05.2023.xlsx',sheet_name='LNG terminals')

### key for version 6 and 7 of the API calls GIE

In [6]:
headers = {"x-key":"d8561648296cb38e6c400823755689941530"} # After July 4th 2022

### Restart here

In [7]:
d1 = '2023-01-01'
d2 = '2023-06-30' # To change in the new year

In [8]:
dates = []
sendOut = []
stored = []
max_storage = []
max_sendout =[]
country = []
Name = []

for item in range(len(Points_APIs)):
        url = Points_APIs.iloc[item,3]+'&from={}&to={}&size=300'.format(d1,d2)
        try:
            r = requests.get(url,headers=headers)
            if r.status_code != 200:
                print(r.status_code)
                print(url)

            raw_data = r.json()
            inner_data = raw_data['data']
            for x in inner_data:
                #here we work with the APIs
                date = x['gasDayStart'] 
                inventory = x['inventory']
                sendOuts = x['sendOut']
                Max_Storage = x['dtmi']
                Max_Sendout = x['dtrs']
                #here we work with the excel file
                countries = Points_APIs.iloc[item,0]
                Names = Points_APIs.iloc[item,5]


                dates.append(date)     
                sendOut.append(sendOuts)
                stored.append(inventory)
                max_storage.append(Max_Storage)
                max_sendout.append(Max_Sendout)
                country.append(countries)
                Name.append(Names)
        except Exception as e:
            print(e)

In [9]:
df = pd.DataFrame()
df['dates'] = dates
df['sendOut'] = sendOut          ## in GWh/d
df['stored'] = stored            ## in 10^3 m^3 LNG
df['Max Storage'] = max_storage  ## in 10^3 m^3 LNG
df['Max Sendout'] = max_sendout  ## in GWh/d
df['country'] = country
df['name'] = Name

In [10]:
# Getting rid of NaNa
df.replace('-', np.NaN,inplace=True)
# Getting rid of observations for which we have no data
df = df.dropna()

In [11]:
df['sendOut'] = pd.to_numeric(df['sendOut'])
df['stored'] = pd.to_numeric(df['stored'])
df['Max Storage'] = pd.to_numeric(df['Max Storage']) 
df['Max Sendout'] = pd.to_numeric(df['Max Sendout']) 

In [12]:
df.head()

,dates,sendOut,stored,Max Storage,Max Sendout,country,name
0,2023-06-18,363.4,430.00,566.0,541.0,BE,Zeebrugge LNG Terminal
1,2023-06-17,380.8,427.07,566.0,541.0,BE,Zeebrugge LNG Terminal
2,2023-06-16,431.4,427.94,566.0,541.0,BE,Zeebrugge LNG Terminal
3,2023-06-15,352.1,407.70,566.0,541.0,BE,Zeebrugge LNG Terminal
4,2023-06-14,312.4,424.21,566.0,541.0,BE,Zeebrugge LNG Terminal


In [13]:
from datetime import date
df.to_csv('raw_data/2023_1.csv')

###  merge with historic data

In [14]:
# Change current directory
import os
os.chdir(Share_point + '\\raw_data')

In [15]:
df_5a = pd.read_csv('2019_1.csv',index_col=0)
df_5b = pd.read_csv('2019_2.csv',index_col=0)
df_6a = pd.read_csv('2020_1.csv',index_col=0)
df_6b = pd.read_csv('2020_2.csv',index_col=0)
df_7a = pd.read_csv('2021_1.csv',index_col=0)
df_7b = pd.read_csv('2021_2.csv',index_col=0)
df_8a = pd.read_csv('2022_1.csv',index_col=0)
df_8b = pd.read_csv('2022_2.csv',index_col=0)
df_9a = pd.read_csv('2023_1.csv',index_col=0)

In [16]:
df = pd.concat([df_5a,df_5b,df_6a,df_6b,df_7a,df_7b,df_8a,df_8b,df_9a])

In [17]:
# del df['Unnamed: 0']
df = df.drop_duplicates()

In [18]:
from datetime import date
today = date.today()
last_obs = today - timedelta(days=2)

In [19]:
df[df['dates']==str(last_obs)].head()

,dates,sendOut,stored,Max Storage,Max Sendout,country,name
0,2023-06-18,363.4,430.00,566.0,541.0,BE,Zeebrugge LNG Terminal
169,2023-06-18,79.5,92.20,140.0,85.5,HR,Krk LNG Terminal (FSRU)
338,2023-06-18,39.4,65.16,80.0,95.9,FR,Fos Tonkin LNG Terminal
507,2023-06-18,256.0,109.75,360.0,337.0,FR,Montoir de Bretagne LNG Terminal
676,2023-06-18,391.5,176.33,570.0,544.8,FR,Dunkerque LNG Terminal


In [20]:
df = df.set_index(pd.DatetimeIndex(df['dates']))

In [21]:
del df['dates']

In [22]:
df = df.sort_values(by='dates')

In [23]:
df

,sendOut,stored,Max Storage,Max Sendout,country,name
dates,,,,,,
2019-01-01,106.3,669.32,758.03,542.9,ES,Barcelona LNG Terminal
2019-01-01,32.9,53.43,80.00,95.9,FR,Fos Tonkin LNG Terminal
2019-01-01,127.2,176.57,330.00,300.0,FR,Fos Cavaou LNG Terminal
2019-01-01,258.9,258.53,540.00,461.0,NL,Rotterdam Gate Terminal
2019-01-01,86.5,197.80,225.00,205.5,GR,Revythoussa LNG Terminal
...,...,...,...,...,...,...
2023-06-18,97.2,134.82,176.23,155.8,DE,Ostsee LNG Terminal / Lubmin (FSRU)
2023-06-18,75.4,49.66,75.00,119.5,IT,Panigaglia LNG Terminal
2023-06-18,155.2,182.95,365.00,269.9,GR,Revythoussa LNG Terminal


In [24]:
# Change current directory
import os
os.chdir(Share_point)

In [25]:
df_C = df.groupby(['dates','country']).sum().reset_index()

C:\Users\giovanni.sgaravatti\AppData\Local\Temp\ipykernel_35544\146117598.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_C = df.groupby(['dates','country']).sum().reset_index()


In [26]:
df_C = df_C.set_index(pd.DatetimeIndex(df_C['dates']))

In [27]:
from datetime import date
today = date.today()
last_obs = today - timedelta(days=2)
print(last_obs)
# enddate = datetime.strptime('2022-07-17', "%d-%m-%Y")
enddate = last_obs
startdate = enddate - timedelta(days=30)

2023-06-18


### Weekly data

In [28]:
df_w = df_C.groupby([pd.Grouper(freq='W'),'country']).sum().reset_index()

C:\Users\giovanni.sgaravatti\AppData\Local\Temp\ipykernel_35544\1569378660.py:1: FutureWarning: The default value of numeric_only in DataFrameGroupBy.sum is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_w = df_C.groupby([pd.Grouper(freq='W'),'country']).sum().reset_index()


In [29]:
df_w=df_w.set_index(pd.DatetimeIndex(df_w['dates']))

In [30]:
df_w.country.unique()

array(['BE', 'ES', 'FR', 'GB', 'GR', 'IT', 'LT', 'NL', 'PL', 'PT', 'ES*',
       'GB*', 'HR', 'DE'], dtype=object)

In [31]:
df_w = df_w[~df_w.country.isin(['ES*','GB','GB*'])] # Get rid of UK and ES TVB aggregate for which there are no data pre-2020

In [32]:
del df_w['dates']

In [33]:
## Can change from new year
data = pd.DataFrame()
data['week'] = list(range(0,54,1))

# plot = df_w.groupby([pd.Grouper(freq='W'),'country']).sum()[:] ## use a converter if you want to transform the data
df_w['week'] = df_w.index.isocalendar().week

hist = df_w['2019-01-05':'2022-01-02'].groupby(['country','week']).mean().reset_index()
values2021 = df_w.loc['2019-12-28':'2022-01-02'].groupby(['country','week']).max().reset_index()
values2022 = df_w.loc['2022-01-03':'2023-01-01'].groupby(['country','week']).max().reset_index()
values2023 = df_w.loc['2023-01-01':'2023-12-31'].groupby(['country','week']).max().reset_index()

In [34]:
hist['regas %'] = (hist['sendOut']/hist['Max Sendout'])*100
values2021['regas %'] = (values2021['sendOut']/values2021['Max Sendout'])*100
values2022['regas %'] = (values2022['sendOut']/values2022['Max Sendout'])*100
values2023['regas %'] = (values2023['sendOut']/values2023['Max Sendout'])*100

In [35]:
import os
os.chdir(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Data\LNG')

In [36]:
Excelwriter = pd.ExcelWriter(r"GIE ALSI LNG terminals\LNG_GIE_weekly_data {}.xlsx".format(today),engine="xlsxwriter")
hist.to_excel(Excelwriter, sheet_name="2019-2021", index=False)
values2021.to_excel(Excelwriter, sheet_name="2021", index=False)
values2022.to_excel(Excelwriter, sheet_name="2022", index=False)
values2023.to_excel(Excelwriter, sheet_name="2023", index=False)
Excelwriter.close()
Excelwriter.save()

C:\Users\giovanni.sgaravatti\AppData\Local\Temp\ipykernel_35544\1029837468.py:7: FutureWarning: save is not part of the public API, usage can give unexpected results and will be removed in a future version
  Excelwriter.save()
c:\Users\giovanni.sgaravatti\anaconda3\lib\site-packages\xlsxwriter\workbook.py:339: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
